<a href="https://colab.research.google.com/github/Davidson-Open-University/phy107-labs/blob/main/notebooks/PHY107-Lab10-Integrated-Investigation.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PHY 107 Lab 10: Integrated Investigation

**Unit 10: Applying mechanics to solve practical problems** · General Practical Physics I · Davidson Open University Global

**In this lab you will:**
- apply energy, motion and power equations to practical problems;
- combine uncertainties in a calculated result;
- use a straight-line graph to find a hidden quantity;
- plan and report a complete investigation from an online lab.

**Time:** about 60 minutes in four parts:
- **Part A – Explore** (about 10 minutes): run the online lab and the examples, and read the explanations.
- **Part B – Practise** (about 20 minutes): fill in the blanks. A self-check tells you if you are right.
- **Part C – Build a tool** (about 20 minutes): complete a short analysis function and test it on data.
- **Part D – Your own data and reflection** (about 10 minutes): analyse readings you took in the online lab (or at home), and write short answers.

**How to use this notebook**
1. Run each code cell in order: click it and press **Shift + Enter** (or the ▶ button).
2. Where you see `...`, replace it with your own code, run the cell, then run the **check** cell under it.
   - ✅ means correct. ❌ gives you a hint: try again. ⏳ means you have not filled in the blank yet.
   - The checks never show the answer, so you learn by working it out. You can try as many times as you like.
3. Stuck? Read the hint, look back at the study session, or ask in the discussion forum. Do not skip ahead: each part builds on the last.
4. Save your work: in Colab choose **File → Save a copy in GitHub** and save it to your own `phy107-labs` repository. Offline in Jupyter, press **Ctrl + S** and later upload the file with **Add file → Upload files** on GitHub.

**Honest data:** keep your own readings separate from the example data, and label every simulated or supplied reading as such. Never invent readings.

*This lab is practice to help you learn and prepare for the unit assignment (lab report). It is not graded unless your tutor tells you otherwise.*

In [ ]:
#@title ▶ Run this cell first: it sets up the self-checks (you do not need to read it)
# You do not need to read or change this code.
import hashlib as _h
class _Blank(Exception): pass
def _norm(x, unordered=False, dp=6):
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray): x = x.tolist()
        elif isinstance(x, _np.bool_): x = bool(x)
        elif isinstance(x, _np.integer): x = int(x)
        elif isinstance(x, _np.floating): x = float(x)
    except ImportError:
        pass
    if x is Ellipsis or x is None: raise _Blank()
    if isinstance(x, bool): return 'b:' + str(x)
    if isinstance(x, (int, float)): return 'n:' + repr(round(float(x), dp) + 0.0)
    if isinstance(x, str): return 't:' + ' '.join(x.lower().split())
    if isinstance(x, (set, frozenset)): return 's{' + ','.join(sorted(_norm(i, False, dp) for i in x)) + '}'
    if isinstance(x, (list, tuple)):
        items = [_norm(i, False, dp) for i in x]
        if unordered: items.sort()
        return 'l[' + ','.join(items) + ']'
    if isinstance(x, dict): return 'd{' + ','.join(sorted(_norm(k, False, dp) + '=' + _norm(v, False, dp) for k, v in x.items())) + '}'
    try:
        import sympy as _sp
        if isinstance(x, _sp.Basic):
            x = _sp.sympify(x)
            if x.is_number: return 'n:' + repr(round(float(x), dp) + 0.0)
            syms = sorted(x.free_symbols, key=str)
            vals = []
            for t in (0.3, 0.7, 1.3):
                v = complex(x.subs({s: t + 0.11 * i for i, s in enumerate(syms)}).evalf())
                vals.append(round(v.real, 6) + 0.0)
            return 'e:' + repr(vals)
    except ImportError:
        pass
    return 'r:' + repr(x)
def _hash(q, s): return _h.sha256(('phy107|' + q + '|' + s).encode()).hexdigest()[:16]
def check(q, value):
    """Check your answer to question q. Prints ✅, ❌ with a hint, or ⏳ if the blank is not filled yet."""
    spec = _CHECKS[q]
    try:
        if callable(value) and 'args' in spec:
            outs = []
            for a in spec['args']:
                try:
                    outs.append(value(*a))
                except _Blank:
                    raise
                except Exception as e:
                    print(f'❌ Question {q}: your function stopped with an error: {type(e).__name__}: {e}')
                    return
            if all(o is None or o is Ellipsis for o in outs): raise _Blank()
            got = _norm(outs, False, spec.get('dp', 6))
        elif spec.get('logic'):
            if value is None or value is Ellipsis: raise _Blank()
            import sympy as _sp
            from itertools import product as _p
            syms = sorted(_sp.sympify(value).free_symbols, key=str)
            got = 'L' + ''.join('1' if bool(_sp.sympify(value).subs(dict(zip(syms, v)))) else '0' for v in _p([True, False], repeat=len(syms))) + ':' + ','.join(map(str, syms))
        else:
            got = _norm(value, spec.get('unordered', False), spec.get('dp', 6))
    except _Blank:
        print(f'⏳ Question {q}: fill in the blank (replace ...) and run the cell above, then run this cell again.')
        return
    if _RECORD is not None:
        _RECORD[q] = _hash(q, got); print(f'✅ Question {q}: recorded.'); return
    if _hash(q, got) == spec['h']:
        print(f'✅ Question {q}: correct! ' + spec.get('ok', 'Well done.'))
    else:
        print(f'❌ Question {q}: not yet. Hint: ' + spec['hint'])
_RECORD = None

_CHECKS = {
    '10.1': {'h': '4d5bc11e0caee7bd', 'hint': 'R = v**2 * np.sin(np.radians(2 * θ)) / 9.81', 'dp': 2},
    '10.2': {'h': '57ed12ceae2d9c50', 'hint': 'v = np.sqrt(2 * g * h); the mass cancels.', 'ok': 'Check it in Energy Skate Park.', 'dp': 2},
    '10.3': {'h': '0f18f04486c338e9', 'hint': 'P = m g h / t.', 'dp': 1},
    '10.4': {'h': '5c58205646b48122', 'hint': 'g = 4π²L/T²; Δg = g × (ΔL/L + 2ΔT/T). The period counts twice because it is squared.', 'ok': 'So g = 9.80 ± 0.08 m/s²: the timing dominates the uncertainty.', 'dp': 2},
    '10.5': {'h': 'baf49d4d18dd8248', 'hint': 'gradient = np.polyfit(np.sin(np.radians(2 * angle)), R, 1)[0]; v = np.sqrt(gradient * 9.81).', 'ok': 'One graph gives the launch speed from all six readings.', 'dp': 1},
    '10.6': {'h': '71babb7963d97042', 'hint': 'm, c = np.polyfit(x, y, 1); return (round(float(m), 3), round(float(c), 3)).', 'ok': 'You now have a general analysis tool for the final practical.', 'dp': 3, 'args': [([0, 1, 2, 3], [1, 3, 5, 7]), ([0.2, 0.4, 0.6, 0.8, 1.0], [0.81, 1.61, 2.42, 3.22, 4.03]), ([1, 2, 3, 4], [2.1, 3.9, 6.2, 7.8])]},
}
print('Self-checks ready. Now carry on with Part A.')

## Part A – Explore (about 10 minutes)
### 🔬 Online lab for this unit (free open educational resources)
- **[PhET Projectile Motion](https://phet.colorado.edu/en/simulations/projectile-motion)** – launch objects at different angles and speeds and measure the range.
- **[PhET Energy Skate Park: Basics](https://phet.colorado.edu/en/simulations/energy-skate-park-basics)** – track kinetic and potential energy as a skater moves.

Open the online lab in a new tab, try it for a few minutes, then come back here. **Label every reading you take there as "simulated".**

**Key results.** Range on level ground: R = v² sin 2θ / g. Energy: mgh = ½mv², so v = √(2gh). Power: P = work / time = mgh / t. For a pendulum result g = 4π²L/T², the fractional uncertainties combine as Δg/g = ΔL/L + 2ΔT/T.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
v, g = 15.0, 9.81
for angle in [30, 45, 60]:
    print(f'{angle}°: range = {v**2 * np.sin(np.radians(2 * angle)) / g:.2f} m')

## Part B – Practise (about 20 minutes)
**B1.** Find the range (m) of a projectile launched at 10 m/s and 45° on level ground (2 decimal places).

In [ ]:
answer_10_1 = ...

In [ ]:
check('10.1', answer_10_1)

In [ ]:
answer_10_2 = ...   # speed (m/s) of a skater at the bottom after dropping 5.0 m (no friction), 2 decimal places

In [ ]:
check('10.2', answer_10_2)

In [ ]:
answer_10_3 = ...   # power (W) to lift 20 kg through 3.0 m in 4.0 s, 1 decimal place

In [ ]:
check('10.3', answer_10_3)

**B4. Uncertainty in g.** A pendulum has L = 0.800 ± 0.002 m and T = 1.795 ± 0.005 s. Find `(g, Δg)` in m/s², to 2 and 2 decimal places.

In [ ]:
g_val = ...
dg = ...
answer_10_4 = (g_val, dg)

In [ ]:
check('10.4', answer_10_4 if ... not in answer_10_4 else ...)

**B5. A hidden quantity from a graph.** In Projectile Motion, the range was measured at several angles for the same launch speed (illustrative data). A graph of R against sin 2θ has gradient v²/g. Find the launch speed v (m/s, 1 decimal place).

In [ ]:
angle = np.array([15, 25, 35, 45, 55, 65])             # degrees (illustrative)
R = np.array([11.5, 17.5, 21.6, 22.9, 21.5, 17.6])     # m
plt.plot(np.sin(np.radians(2 * angle)), R, 'o'); plt.xlabel('sin 2θ'); plt.ylabel('range R (m)'); plt.show()

In [ ]:
answer_10_5 = ...

In [ ]:
check('10.5', answer_10_5)

## Part C – Build a tool (about 20 minutes)
Complete `line_fit(x, y)`. It should return a tuple **(gradient, intercept)** from `np.polyfit(x, y, 1)`, both rounded to 3 decimal places. This general tool works for every straight-line analysis in the course.

In [ ]:
def line_fit(x, y):
    ...      # YOUR CODE

In [ ]:
check('10.6', line_fit)

## Part D – Your own investigation (about 10 minutes, then continue for your assignment)
**D1. Prepares you for the Unit 10 assignment and the final practical.** Choose one question you can test in an online lab from this course, for example "How does launch angle affect range?" or "How does length affect a pendulum's period?". Record:
1. your question and prediction;
2. the independent, dependent and controlled variables;
3. at least six readings, labelled "simulated";
4. a straight-line graph and its gradient, found with `line_fit`;
5. your conclusion, with an uncertainty and one limitation.

In [ ]:
my_x = [...]
my_y = [...]
# print(line_fit(my_x, my_y))

**D2. Reflect.**
1. Which part of your investigation contributed the largest uncertainty?
2. What can a simulation not tell you that a real experiment would?
3. How would you improve your method if you repeated it?

*Type your answer here (double-click this cell to edit it).*

## Finished? Save your work
In Colab: **File → Save a copy in GitHub** → your `phy107-labs` repository → commit message *"Lab 10 completed"*.
Offline: save with **Ctrl + S**, then upload the file to your repository with **Add file → Upload files** when you are online.

Then go back to Moodle and mark the lab as done.